# Qiskit Fall Fest 2026 — Phase 2
## O3: Knapsack Optimization | Qubit Crew

This notebook validates the classical knapsack baseline, checks processor configuration files, audits project modules, and reports whether the project is ready for quantum benchmarking.

**Problem:** capacity 15; weights [5, 3, 4, 2, 6]; values [10, 4, 7, 5, 8].

**Noise specification:** single-qubit depolarizing probability 0.001; two-qubit depolarizing probability 0.005; readout bit-flip probability 0.01.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import sys
import importlib
import traceback

print('Python:', sys.version.split()[0])
print('Working directory:', Path.cwd())
print('Notebook initialization complete.')

## 1. Define the official knapsack instance

The following values are the fixed problem inputs. This cell does not depend on any custom project modules.

In [ ]:
ITEMS = ['A', 'B', 'C', 'D', 'E']
WEIGHTS = [5, 3, 4, 2, 6]
VALUES = [10, 4, 7, 5, 8]
CAPACITY = 15

assert len(ITEMS) == len(WEIGHTS) == len(VALUES) == 5
assert all(w > 0 for w in WEIGHTS)
assert all(v >= 0 for v in VALUES)

print('Items:', ITEMS)
print('Weights:', WEIGHTS)
print('Values:', VALUES)
print('Capacity:', CAPACITY)

## 2. Exhaustive classical baseline

Enumerate all 2^5 possible selections and find the best feasible solution. This is an exact reference against which the quantum results can later be checked.

In [ ]:
all_solutions = []

for bits in itertools.product([0, 1], repeat=len(ITEMS)):
    weight = sum(b * w for b, w in zip(bits, WEIGHTS))
    value = sum(b * v for b, v in zip(bits, VALUES))
    all_solutions.append({
        'bits': list(bits),
        'items': [ITEMS[i] for i, b in enumerate(bits) if b],
        'weight': weight,
        'value': value,
        'feasible': weight <= CAPACITY
    })

feasible_solutions = [s for s in all_solutions if s['feasible']]
best_value = max(s['value'] for s in feasible_solutions)
optimal_solutions = [s for s in feasible_solutions if s['value'] == best_value]
best_solution = min(optimal_solutions, key=lambda s: s['weight'])

print('Total combinations:', len(all_solutions))
print('Feasible combinations:', len(feasible_solutions))
print('Optimal items:', best_solution['items'])
print('Optimal bit vector (A to E):', best_solution['bits'])
print('Optimal weight:', best_solution['weight'])
print('Optimal value:', best_solution['value'])

assert len(all_solutions) == 32
assert best_solution['items'] == ['A', 'B', 'C', 'D']
assert best_solution['weight'] == 14
assert best_solution['value'] == 26
print('\nPASS: Classical baseline matches the expected optimum.')

## 3. Noise parameters

Readout error is a probability of 0.01 (1%), not the number 1. These constants document the specified benchmark settings; the actual noise model must use them when constructing simulator errors.

In [ ]:
NOISE = {
    'single_qubit_depolarizing': 0.001,
    'two_qubit_depolarizing': 0.005,
    'readout_bit_flip': 0.01
}

assert 0 <= NOISE['single_qubit_depolarizing'] <= 1
assert 0 <= NOISE['two_qubit_depolarizing'] <= 1
assert 0 <= NOISE['readout_bit_flip'] <= 1
print(json.dumps(NOISE, indent=2))

## 4. Locate project files

This checks common project locations without changing or deleting files. If this notebook is opened directly from GitHub in Colab, the repository source files may need to be cloned or uploaded into the runtime first.

In [ ]:
SEARCH_ROOTS = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive')]
found_src = []

for root in SEARCH_ROOTS:
    if root.exists():
        direct = root / 'src'
        if direct.is_dir() and direct not in found_src:
            found_src.append(direct)
        try:
            for candidate in root.glob('*/src'):
                if candidate.is_dir() and candidate not in found_src:
                    found_src.append(candidate)
        except PermissionError:
            pass

if found_src:
    print('Found src directories:')
    for p in found_src:
        print(' -', p)
    PROJECT_ROOT = found_src[0].parent
    print('\nSelected project root:', PROJECT_ROOT)
else:
    PROJECT_ROOT = None
    print('WARNING: No project src/ directory found in the checked locations.')
    print('The independent classical baseline above still works.')

## 5. Audit source modules

Check which files and functions exist before attempting imports. The project uses the British spelling `visualisation.py`; do not assume `visualization.py` exists.

In [ ]:
EXPECTED_FILES = [
    'src/__init__.py',
    'src/problem.py',
    'src/processors.py',
    'src/metrics.py',
    'src/routing.py',
    'src/visualisation.py',
    'requirements.txt'
]

if PROJECT_ROOT is None:
    print('SKIPPED: project files are not present in this runtime.')
else:
    for name in EXPECTED_FILES:
        status = 'FOUND' if (PROJECT_ROOT / name).is_file() else 'MISSING'
        print(f'{status:7} {name}')

    print('\nProcessor configuration JSON files:')
    processor_dir = PROJECT_ROOT / 'processors'
    json_files = sorted(processor_dir.glob('*.json')) if processor_dir.is_dir() else []
    if json_files:
        for p in json_files:
            print(' -', p.name)
    else:
        print('No JSON files found in processors/.')

## 6. Validate processor configurations

Processor A must have 5 qubits and 4 edges. Processor B must have 9 qubits and 12 edges. The complete edge lists must come from the actual configuration files; this notebook will not invent missing topology data.

In [ ]:
EXPECTED_PROCESSORS = {
    'A': {'qubits': 5, 'edges': 4},
    'B': {'qubits': 9, 'edges': 12}
}

def find_processor_jsons(root):
    if root is None:
        return []
    candidates = []
    for folder in [root / 'processors', root / 'data']:
        if folder.is_dir():
            candidates.extend(folder.rglob('*.json'))
    return sorted(set(candidates))

def inspect_processor_json(path):
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
        print(f'\n{path.name}: valid JSON')
        print(json.dumps(data, indent=2)[:2500])
        return data
    except Exception as exc:
        print(f'\n{path.name}: INVALID JSON — {exc}')
        return None

processor_jsons = find_processor_jsons(PROJECT_ROOT)
processor_data = {}
for path in processor_jsons:
    name = path.stem.lower()
    if 'processor_a' in name or name in ('a', 'processor a'):
        processor_data['A'] = inspect_processor_json(path)
    elif 'processor_b' in name or name in ('b', 'processor b'):
        processor_data['B'] = inspect_processor_json(path)

if not processor_data:
    print('Processor A/B JSON files were not identified. No topology validation was performed.')
else:
    print('\nFound processor files:', sorted(processor_data))
    print('Note: field names vary between projects; verify the printed JSON against the official configuration.')

print('\nExpected hardware summary:', EXPECTED_PROCESSORS)

## 7. Check project API compatibility

The earlier notebook expected several functions that may not exist in the current files. This cell reports import failures and missing functions instead of crashing the entire notebook.

In [ ]:
REQUIRED_APIS = {
    'src.problem': [
        'load_instance', 'brute_force', 'make_encoding',
        'optimize_qaoa', 'build_qaoa_circuit', 'bits_to_items',
        'qaoa_probabilities'
    ],
    'src.processors': [
        'load_processor', 'build_noise_model',
        'noise_parameters', 'processor_summary'
    ],
    'src.routing': [
        'compile_all_to_all', 'run_case',
        'compile_and_run_noisy', 'OPT_LEVEL'
    ],
    'src.metrics': [
        'circuit_metrics', 'case_row', 'aggregate', 'save_table',
        'build_report', 'update_readme_results', 'df_to_markdown',
        'solution_metrics', 'std_error'
    ],
    'src.visualisation': [
        'plot_processor_graph', 'draw_circuit',
        'plot_logical_vs_transpiled', 'plot_ab_performance',
        'plot_distribution', 'plot_tradeoffs', 'plot_landscape',
        'COL_A', 'COL_B'
    ]
}

if PROJECT_ROOT is None:
    print('SKIPPED: source directory unavailable.')
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    for module_name, required_names in REQUIRED_APIS.items():
        print(f'\n[{module_name}]')
        try:
            module = importlib.import_module(module_name)
            missing = [name for name in required_names if not hasattr(module, name)]
            if missing:
                print('MISSING FUNCTIONS/CONSTANTS:', ', '.join(missing))
            else:
                print('All expected names are available.')
        except Exception as exc:
            print('IMPORT FAILED:', type(exc).__name__, str(exc))
            print('Fix this module before running full quantum benchmarks.')

## 8. Reproducibility summary

This section saves the independently verified classical baseline in memory and displays the benchmark settings. It does not overwrite project files or claim that QAOA simulations have completed.

In [ ]:
baseline = {
    'problem': 'O3 Knapsack',
    'capacity': CAPACITY,
    'items': ITEMS,
    'weights': WEIGHTS,
    'values': VALUES,
    'total_combinations': len(all_solutions),
    'feasible_combinations': len(feasible_solutions),
    'optimal_items': best_solution['items'],
    'optimal_bits_A_to_E': best_solution['bits'],
    'optimal_weight': best_solution['weight'],
    'optimal_value': best_solution['value'],
    'noise_parameters': NOISE
}

print(json.dumps(baseline, indent=2))
print('\nCLASSICAL BASELINE: PASS')
print('FULL QUANTUM BENCHMARK: NOT RUN by this baseline notebook.')
print('Next step: resolve missing module APIs, then add and run the QAOA A/B experiments.')